# Wstępna Eksploracyjna Analiza Danych ze Steam
Celem tej sekcji jest ustalenie wstępnych wymagań wobec zbioru danych przed jego pełnym pobraniem ze Steam API.

W projekcie celem jest stworzenie algorytmu wykrywającego sentyment oceny gry, uwzględniając sarkzam.

Wytrenowanie skutecznego modelu będzie wymagało przekroju różnych opinii, z różnych gatunków gier, o różnych wymogach technicznych. Powody opinii negatywnej czy sarkastycznej mogą wynikać zarówno z problemów z fabułą i mechaniką gry, ale również z braku optymalizacji gry dla słabszego sprzętu. Algorytm wytrenowany wyłącznie na recenzjach najpopularniejszych gier mogłoby skutkować z tendencyjnością algorytmu i wysoką liczebnością klasy False Negative.

Dodatkowo zdaniem autora najpopularniejsze gry mogą częściej mieć problemy z optymalizacją niż z samą rozgrywką, niż przeciętna gra. Gracze chętniej spróbują zagrać w ciekawą grę, która jest niezoptymalizowana na ich sprzęt, z nadzieją, że twórca gry po pewnym czasie wypuści aktualizację, która ustabilizuje ilość klatek na sekundę na ich sprzęcie. Natomiast, jeśli gra jest dobrze zoptymalizowana ale nudna gra nigdy nie trafi do listy najpopularniejszych gier.

Osobnym wyzwaniem jest ilość opinii w języku polskim. Sarkazm jest zjawiskiem bardzo specficznym dla każdego języka, dlatego do trenowania modelu autor wybierze wyłącznie opinie w języku polskim. Tym sposobem autor będzie mógł na koniec podejrzeć wyniki algorytmu i samodzielnie ocenić jego poprawne funkcjonowanie. Ograniczenie językowe oznacza, że skupienie się wyłącznie na mało popularnych grach może skutkować trudnością z uzbieraniem odpowiedniej ilości opinii i znacznie większą ilością zapytań do Steam API.

Zważając na powyższe ograniczenia autor zamierza najpierw podzielić dużą populację najczęściej granych gier podzielić na grupy według ich kategorii oceny Steam, a następnie z każdej grupy losowo wybrać podobną liczbę gier, aby cały zbiór recenzji posiadał około **??** tysięcy opinii.

## Przegląd najpopularniejszych gier na platformie Steam

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from datetime import datetime
from zoneinfo import ZoneInfo

In [23]:
url = "https://store.steampowered.com/search/results/?supportedlang=polish%2Cenglish&category1=998&ndl=1&start=100"
    
# Udajemy prawdziwą przeglądarkę (Header), żeby Steam nas nie zablokował
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept-Language': 'pl,en;q=0.9'
}

# 4. Wywołujemy zapytanie przekazując ZARÓWNO nagłówki, jak i ciasteczka
response = requests.get(url, headers=headers)

# Jeśli strona nie zwraca JSON-a, bierzemy surowy tekst HTML
html_content = response.text

# print(html_content)
soup = BeautifulSoup(html_content, 'html.parser')

search_results = soup.find_all('a', class_='search_result_row')
# print(search_results)
n_games = 0
for item in search_results:
    # try:
    # Wyciągamy ID aplikacji (Steam trzyma je w atrybucie 'data-ds-appid')
    app_id = int(item.get('data-ds-appid'))
    
    # Wyciągamy nazwę gry
    name = item.find('span', class_='title').text.strip()
    
    # Szukamy sekcji z opiniami (ma klasą 'search_review_summary')
    review_div = item.find('span', class_='search_review_summary')
    
    if review_div:
        # Wyciągamy ukryty opis tekstowy, np. "Mixed - 45% of the 1,200 user reviews..."
        review_html = review_div.get('data-tooltip-html', '')
        
        # Używamy Wyrażeń Regularnych (Regex), żeby wyciągnąć procenty i liczbę opinii
        # Szukamy wzorca: "XX% of the XX,XX user reviews"
        percent_grp = re.findall(r'(\d\d)%', review_html)
        reviews_grp = re.findall(r'z ([\d,]+) recenzji', review_html)
        
        percent_positive = int(percent_grp[0]) if percent_grp else None
        
        # Usuwamy przecinki z liczby opinii, np. 1,200 -> 1200
        total_reviews = int(reviews_grp[0].replace(',', '')) if reviews_grp else 0
        print(f"{app_id} - {name} - {percent_positive}% - {total_reviews}")
        n_games += 1

print(n_games)
    # except Exception as e:
    #     # Jeśli jedna gra rzuci błędem podczas parsowania, idziemy do kolejnej
    #     continue

1643320 - S.T.A.L.K.E.R. 2: Heart of Chornobyl - 71% - 1944
2863680 - ZERO PARADES: For Dead Spies - 76% - 670
4069520 - Super Battle Golf - 94% - 7338
2707930 - Palia - 87% - 444
2357570 - Overwatch® - 42% - 4317
376210 - The Isle - 89% - 3456
1966720 - Lethal Company - 97% - 7704
1716740 - Starfield - 57% - 584
1149460 - ICARUS - 81% - 495
3326230 - Hozy - 80% - 2877
3526710 - Everything is Crab: The Animal Evolution Roguelite - 76% - 4470
518790 - theHunter: Call of the Wild™ - 95% - 5626
393380 - Squad - 87% - 1150
3552140 - Retro Rewind - Video Store Simulator - 96% - 7375
1599340 - Lost Ark - 68% - 6341
2139460 - Once Human - 77% - 1738
2074920 - The First Descendant - 64% - 939
594570 - Total War: WARHAMMER II - 94% - 847
284160 - BeamNG.drive - 97% - 19764
2622380 - ELDEN RING NIGHTREIGN - 89% - 1175
4025700 - Heartopia - 65% - 13932
2747330 - Species: Unknown - 94% - 6429
294100 - RimWorld - 99% - 3024
1466860 - Age of Empires IV: Anniversary Edition - 87% - 598
3065800 - Mara

In [ ]:
games_list = []      
games_list.append({
    'app_id': app_id,
    'name': name,
    'percent_positive': percent_positive + 1,
    'total_reviews': total_reviews,
    'last_updated': datetime.now(ZoneInfo('Europe/Warsaw'))
})
pd_games = pd.DataFrame(games_list)
print(pd_games)

   app_id          name  percent_positive  total_reviews  \
0  582660  Black Desert                82           1782   

                      last_updated  
0 2026-05-24 22:42:28.404273+02:00  
